# Rowan's model, first iteration

1. **CNN:** 9 convolution layers (ReLU) and 3 max-pooling layers turn a frame (3 × 1088 × 1920) into 64 × 17 × 30.
2. **FC1–FC3 (sigmoid):** for each cell of a 17 × 30 grid: *person here?* and the box `cx, cy, w, h`.
3. **Transformer:** each kept cell's 5 numbers become a 64-number token; 3 transformer encoder layers.
4. **Comparison:** each token of frame t is scored against frame t−1's tokens and a learned "new person" vector
   (dot product ÷ √64, softmax). The most likely option is its identity.

Trained with cross-entropy only, then compared with the DeepSORT baseline on MOT17-13. Input comes from
`Exploratory data analysis.ipynb` (run it first); results go to `outputs/model/`. Needs the GPU.

## M.0 Setup

In [ ]:
import itertools
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence

np.float, np.int = float, int     # TrackEval still uses these two names, removed from NumPy (they were plain aliases)
import trackeval

import mot17_data
from mot17_data import DATA_ROOT, MOT17Frames, MOT17FramePairs, make_loader, boxes_to_pixels

assert torch.cuda.is_available(), "No GPU: install the CUDA build of PyTorch (see requirements.txt), then restart"
DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# ---- Settings ----
BATCH_SIZE = 64            # pairs per Adam step (Adam, default betas, no weight decay)
LEARNING_RATE = 0.001
EPOCHS = 16
MAX_EPOCHS = 24            # every run trains 24 epochs; the grid compares epochs 8, 16 and 24
MICRO_BATCH = 4            # pairs on the GPU at once (gradients are added up until BATCH_SIZE pairs)
ROWS, COLS = 17, 30        # prediction grid: each cell is 64 x 64 pixels of the 1920 x 1088 input
PERSON_THRESHOLD = 0.5
NUM_WORKERS = 8
GRID = list(itertools.product([32, 64, 128], [0.0001, 0.001, 0.01]))     # (batch size, learning rate): 9 runs
# GRID = list(itertools.product([32, 64], [0.0001, 0.001]))              # OPTION if time is short: 4 runs
ALLOW_LONG_GRID = False    # the grid stops to ask Rowan if it would take more than 10 hours

seed_everything()
OUT = mot17_data.PROJECT_ROOT / "outputs" / "model"
for folder in ["checkpoints", "history", "figures", "trackers", "eval"]:
    (OUT / folder).mkdir(parents=True, exist_ok=True)
videos = mot17_data.read("videos").set_index("video")
gt_before = {v: (DATA_ROOT / videos.loc[v, "folder"] / "gt" / "gt.txt").read_bytes() for v in videos.index}

## M.1 Training targets
Each box goes to the grid cell holding its centre: row = ⌊`cy` × 17⌋, column = ⌊`cx` × 30⌋. Nothing is altered; we only choose which boxes are targets:
* **One person per cell:** if two centres share a cell, the more visible box wins (then the larger); the other isn't a target in that frame.
* **Centre outside the frame:** in no cell, so not a target.
* **`w` or `h` above 1:** still *person here?* = 1, but left out of the box loss (a sigmoid can't output more than 1).

In [ ]:
def frame_targets(boxes, ids, visibility):
    """Grid targets of one frame: person [17,30] (0/1), box [17,30,4], box_valid [17,30], cell_id [17,30] (0 = empty),
    and counts of what happened to each box."""
    person = np.zeros((ROWS, COLS), np.float32)
    box = np.zeros((ROWS, COLS, 4))
    box_valid = np.zeros((ROWS, COLS), bool)
    cell_id = np.zeros((ROWS, COLS), np.int64)
    counts = Counter(boxes=len(boxes))
    for k in sorted(range(len(boxes)), key=lambda k: (-visibility[k], -boxes[k][2] * boxes[k][3])):
        cx, cy, w, h = boxes[k]
        if not (0 <= cx < 1 and 0 <= cy < 1):
            counts["centre outside"] += 1
            continue
        r, c = int(cy * ROWS), int(cx * COLS)
        if person[r, c]:
            counts["lost to collision"] += 1
            continue
        person[r, c], box[r, c], cell_id[r, c], box_valid[r, c] = 1, boxes[k], ids[k], w <= 1 and h <= 1
        counts["assigned"] += 1
        counts["left out of box loss"] += not box_valid[r, c]
    return person, box, box_valid, cell_id, counts


def batch_targets(*batches):
    """Targets of every frame of the batches (in order), stacked and on the GPU."""
    targets = [frame_targets(b.numpy(), i.numpy(), v.numpy()) for batch in batches
               for b, i, v in zip(batch["boxes"], batch["ids"], batch["visibility"])]
    person, box, box_valid, cell_id = (torch.tensor(np.stack(x)).to(DEVICE) for x in list(zip(*targets))[:4])
    return person, box.float(), box_valid, cell_id


train_pairs, val_pairs, test_pairs = (MOT17FramePairs(s) for s in ["train", "val", "test"])
target_counts, largest_difference = {}, 0.0
for split, pairs in [("train", train_pairs), ("val", val_pairs), ("test", test_pairs)]:
    total = Counter(frames=len(pairs.frames))
    for key in pairs.frames.frames[["video", "frame"]].itertuples(index=False, name=None):
        b = pairs.frames.boxes.get(key, np.zeros((0, 6)))
        person, box, _, cell_id, counts = frame_targets(b[:, :4], b[:, 4], b[:, 5])
        total += counts
        assigned = person > 0     # check: each target box is exactly its boxes.csv box
        row_of_id = {int(i): k for k, i in enumerate(b[:, 4])}
        if assigned.any():
            original = b[[row_of_id[i] for i in cell_id[assigned]], :4]
            largest_difference = max(largest_difference, np.abs(box[assigned] - original).max())
    target_counts[split] = total
target_counts = pd.DataFrame(target_counts).T.fillna(0).astype(int)
display(target_counts)

t = target_counts.loc["train"]
POS_WEIGHT = torch.tensor((t["frames"] * ROWS * COLS - t["assigned"]) / t["assigned"], device=DEVICE)
print(f"pos_weight = empty cells / person cells in the training targets = {POS_WEIGHT.item():.2f}")

## M.2 The model
Three 4 × 4 poolings take 1088 × 1920 to exactly 17 × 30. Tokens: the ground-truth person cells in training, cells
with *person here?* > 0.5 when tracking. Frames have different numbers of tokens, so they are padded and masked.
The "new person" option lets someone who just appeared get a new ID. (`ReLU(inplace=True)` only saves memory.)

In [ ]:
class TrackerNet(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        for c_in, c_out in [(3, 16), (16, 32), (32, 64)]:
            for k in range(3):
                layers += [nn.Conv2d(c_in if k == 0 else c_out, c_out, kernel_size=3, padding=1), nn.ReLU(inplace=True)]
            layers.append(nn.MaxPool2d(4))
        self.cnn = nn.Sequential(*layers)
        self.fc1 = nn.Linear(64 * ROWS * COLS, 1024)
        self.fc2 = nn.Linear(1024, 1024)
        self.fc3 = nn.Linear(1024, ROWS * COLS * 5)
        self.token = nn.Linear(5, 64)
        self.transformer = nn.ModuleList(
            nn.TransformerEncoderLayer(d_model=64, nhead=4, dim_feedforward=128, dropout=0.0, batch_first=True)
            for _ in range(3))
        self.new_person = nn.Parameter(torch.zeros(64))

    def forward(self, images):
        """Frames [B, 3, 1088, 1920] -> FC3's values before its sigmoid, [B, 17, 30, 5]
        (person here?, cx, cy, w, h per cell). The losses apply the sigmoid inside BCEWithLogitsLoss."""
        x = self.cnn(images).flatten(1)
        x = torch.sigmoid(self.fc2(torch.sigmoid(self.fc1(x))))
        return self.fc3(x).view(-1, ROWS, COLS, 5)

    def encode(self, logits, kept):
        """Each frame's kept cells -> tokens -> transformer. Returns one [n, 64] tensor per frame."""
        counts = kept.sum(dim=(1, 2)).tolist()
        tokens = list(self.token(torch.sigmoid(logits[kept])).split(counts))
        busy = [i for i, n in enumerate(counts) if n > 0]               # frames with at least one token
        if busy:
            x = pad_sequence([tokens[i] for i in busy], batch_first=True)
            lengths = torch.tensor([counts[i] for i in busy], device=x.device)
            padding = torch.arange(x.shape[1], device=x.device) >= lengths[:, None]
            for layer in self.transformer:
                x = layer(x, src_key_padding_mask=padding)
            for j, i in enumerate(busy):
                tokens[i] = x[j, :counts[i]]
        return tokens

    def compare(self, current, previous):
        """Scores of frame t's tokens against [new person, frame t-1's tokens]: dot product / sqrt(64)."""
        return current @ torch.cat([self.new_person[None], previous]).T / 8


# Architecture table: real output shapes from one dummy frame, and the weights of every layer.
weights = lambda module: sum(p.numel() for p in module.parameters())
model = TrackerNet().to(DEVICE)
rows = []
with torch.no_grad():
    x = torch.zeros(1, 3, 1088, 1920, device=DEVICE)
    for layer in model.cnn:
        x = layer(x)
        if not isinstance(layer, nn.ReLU):
            rows.append([type(layer).__name__, tuple(x.shape[1:]), weights(layer)])
    x = x.flatten(1)
    rows.append(["Flatten", tuple(x.shape[1:]), 0])
    for name, fc in [("FC1 + sigmoid", model.fc1), ("FC2 + sigmoid", model.fc2), ("FC3 + sigmoid", model.fc3)]:
        x = torch.sigmoid(fc(x))
        rows.append([name, tuple(x.shape[1:]), weights(fc)])
    t = model.token(x.view(-1, ROWS, COLS, 5)[0, 0, :3])                     # 3 example cells -> 3 tokens
    rows.append(["Tokenization (Linear 5 -> 64)", tuple(t.shape[1:]), weights(model.token)])
    for k, layer in enumerate(model.transformer):
        t = layer(t[None])[0]
        rows.append([f"Transformer encoder {k + 1}", tuple(t.shape[1:]), weights(layer)])
rows.append(["New person vector", (64,), model.new_person.numel()])
architecture = pd.DataFrame(rows, columns=["layer", "output (per frame; per token after FC3)", "weights"])
display(architecture)
print(f"Total weights: {weights(model):,}")

## M.3 Loss and training
**Loss per pair = person loss + box loss + ID loss** (all cross-entropy):
* **Person:** binary cross-entropy on *person here?* over the 510 cells of both frames, with `pos_weight`.
* **Box:** binary cross-entropy on `cx, cy, w, h` of the cells with a valid box target, both frames.
* **ID:** cross-entropy for frame t's tokens; the answer is the same person in frame t−1, or "new person" (option 0).

**Gradient accumulation:** 64 full-size pairs don't fit in 12 GB, so 4 pairs go through at a time and each one's loss
is divided by the pairs in its step; one Adam step then gets exactly the gradient of the whole batch.
Validation and test losses use the same ground-truth cells as training.

In [ ]:
def pair_losses(model, prev, cur):
    """Person, box and ID loss of each pair in a micro-batch (frames t-1 in `prev`, frames t in `cur`),
    and how many of frame t's tokens got the right identity, out of how many."""
    P = len(prev["frame"])
    person_t, box_t, valid_t, id_t = batch_targets(prev, cur)            # frames t-1 first, then frames t
    images = torch.cat([prev["image"], cur["image"]]).to(DEVICE, non_blocking=True)
    assert images.is_cuda, "the batch is not on the GPU"
    logits = model(images)

    person = F.binary_cross_entropy_with_logits(logits[..., 0], person_t, pos_weight=POS_WEIGHT,
                                                reduction="none").mean(dim=(1, 2))
    person = (person[:P] + person[P:]) / 2
    box = F.binary_cross_entropy_with_logits(logits[..., 1:], box_t, reduction="none").mean(dim=-1) * valid_t
    box_sum, n_valid = box.sum(dim=(1, 2)), valid_t.sum(dim=(1, 2))
    box = (box_sum[:P] + box_sum[P:]) / (n_valid[:P] + n_valid[P:]).clamp(min=1)

    kept = person_t > 0                                   # training tokens: the ground-truth person cells
    tokens = model.encode(logits, kept)
    ids = [id_t[f][kept[f]].tolist() for f in range(2 * P)]
    id_loss, correct, total = [], 0, 0
    for p in range(P):
        prev_ids, cur_ids = ids[p], ids[P + p]
        if not cur_ids:
            id_loss.append(torch.zeros((), device=DEVICE))
            continue
        answer = torch.tensor([prev_ids.index(i) + 1 if i in prev_ids else 0 for i in cur_ids], device=DEVICE)
        scores = model.compare(tokens[P + p], tokens[p])
        id_loss.append(F.cross_entropy(scores, answer))
        correct += (scores.argmax(dim=1) == answer).sum().item()
        total += len(answer)
    return person, box, torch.stack(id_loss), correct, total


def train_epoch(model, optimizer, batch_size):
    """One pass over the training pairs; one Adam step per batch_size pairs. Returns the mean losses and ID accuracy."""
    model.train()
    n, done, sums, correct, total = len(train_pairs), 0, np.zeros(4), 0, 0
    optimizer.zero_grad()
    for prev, cur in train_loader:
        step_pairs = min(batch_size, n - done // batch_size * batch_size)    # the epoch's last step is smaller
        person, box, ident, c, t = pair_losses(model, prev, cur)
        loss = person + box + ident
        (loss.sum() / step_pairs).backward()
        done += len(loss)
        if done % batch_size == 0 or done == n:
            optimizer.step()
            optimizer.zero_grad()
        sums += [x.sum().item() for x in (loss, person, box, ident)]
        correct, total = correct + c, total + t
    return dict(zip(["loss", "person", "box", "id"], sums / n), id_acc=correct / total)


@torch.no_grad()
def evaluate(model, loader):
    """Mean losses and ID accuracy over the validation or test pairs."""
    model.eval()
    sums, correct, total = np.zeros(4), 0, 0
    for prev, cur in loader:
        person, box, ident, c, t = pair_losses(model, prev, cur)
        sums += [x.sum().item() for x in (person + box + ident, person, box, ident)]
        correct, total = correct + c, total + t
    return dict(zip(["loss", "person", "box", "id"], sums / len(loader.dataset)), id_acc=correct / total)


train_loader = make_loader(train_pairs, shuffle=True, batch_size=MICRO_BATCH, num_workers=NUM_WORKERS)
val_loader = make_loader(val_pairs, shuffle=False, batch_size=MICRO_BATCH, num_workers=NUM_WORKERS)
test_loader = make_loader(test_pairs, shuffle=False, batch_size=MICRO_BATCH, num_workers=NUM_WORKERS)


def train_run(batch_size, lr):
    """Train one setting from scratch for 24 epochs, saving its history and checkpoints at epochs 8, 16 and 24.
    A run that already finished is just read back from its history file."""
    name = f"bs{batch_size}_lr{lr:g}"
    path = OUT / "history" / f"{name}.csv"
    if path.exists() and len(pd.read_csv(path)) == MAX_EPOCHS:
        return pd.read_csv(path)
    seed_everything()
    model = TrackerNet().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        start = time.time()
        row = {"epoch": epoch, **{f"train_{k}": v for k, v in train_epoch(model, optimizer, batch_size).items()},
               **{f"val_{k}": v for k, v in evaluate(model, val_loader).items()},
               **{f"test_{k}": v for k, v in evaluate(model, test_loader).items()}, "seconds": time.time() - start}
        history.append(row)
        pd.DataFrame(history).to_csv(path, index=False)
        if epoch in (8, 16, 24):
            torch.save(model.state_dict(), OUT / "checkpoints" / f"{name}_ep{epoch}.pt")
        print(f"{name} epoch {epoch}: loss train {row['train_loss']:.4f} val {row['val_loss']:.4f}, "
              f"ID accuracy val {row['val_id_acc']:.3f}, {row['seconds']:.0f} s, "
              f"GPU memory peak {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    return pd.DataFrame(history)


def load_model(batch_size, lr, epochs):
    model = TrackerNet().to(DEVICE)
    model.load_state_dict(torch.load(OUT / "checkpoints" / f"bs{batch_size}_lr{lr:g}_ep{epochs}.pt"))
    return model.eval()

## M.4 Tracking and scoring
Each video runs frame by frame in order, remembering only the previous frame's tokens and IDs. On a video's first
frame everyone is new. **One ID per person per frame:** each box takes its most likely option; if two boxes choose
the same previous person, the more probable one keeps it and the other takes its next most likely option (this
greedy rule only stops duplicate IDs, which TrackEval rejects). Scored with the same TrackEval settings as the baseline.

In [ ]:
def assign_ids(prob):
    """prob [n, 1 + m] -> chosen option per box (0 = new person, j = previous person j)."""
    prob, choice = prob.clone(), [None] * len(prob)
    for _ in range(len(prob)):
        best, option = prob.max(dim=1)
        best[[i for i, c in enumerate(choice) if c is not None]] = -1      # boxes already decided
        i = int(best.argmax())
        choice[i] = int(option[i])
        if choice[i] > 0:
            prob[:, choice[i]] = -1                                         # that previous person is taken
    return choice


@torch.no_grad()
def track(model, split, name):
    """Track every video of a split; write MOTChallenge files to trackers/<name>/data/. Returns every output box."""
    model.eval()
    rows = []
    for batch in make_loader(MOT17Frames(split), shuffle=False, batch_size=8, num_workers=NUM_WORKERS):
        images = batch["image"].to(DEVICE)
        assert images.is_cuda, "the batch is not on the GPU"
        logits = model(images)
        kept = torch.sigmoid(logits[..., 0]) > PERSON_THRESHOLD
        tokens = model.encode(logits, kept)
        for f, (video, frame) in enumerate(zip(batch["video"], batch["frame"])):
            if frame == 1:                                    # a new video: no previous frame
                previous, previous_ids, next_id = tokens[f][:0], [], 1
            cells = torch.sigmoid(logits[f][kept[f]]).cpu().numpy()          # person?, cx, cy, w, h
            choice = assign_ids(torch.softmax(model.compare(tokens[f], previous), dim=1).cpu()) if len(cells) else []
            ids = []
            for c in choice:
                if c == 0:
                    ids.append(next_id)
                    next_id += 1
                else:
                    ids.append(previous_ids[c - 1])
            rows += [(video, frame, i, *cell) for i, cell in zip(ids, cells)]
            previous, previous_ids = tokens[f], ids
    found = pd.DataFrame(rows, columns=["video", "frame", "id", "conf", "cx", "cy", "w", "h"])
    for video in videos.index[videos["split"] == split]:
        b = found[found["video"] == video]
        px = boxes_to_pixels(b, videos.loc[video, "width"], videos.loc[video, "height"])
        lines = [f"{r.frame},{r.id},{p.left:.2f},{p.top:.2f},{p.width:.2f},{p.height:.2f},{r.conf:.4f},-1,-1,-1"
                 for r, p in zip(b.itertuples(), px.itertuples())]
        path = OUT / "trackers" / name / "data" / f"{video}-FRCNN.txt"
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text("\n".join(lines) + "\n")
    return found


def score(tracker_names, video_names=("MOT17-10", "MOT17-13")):
    """HOTA, MOTA, IDF1, ID switches, false positives and false negatives (same settings as the baseline)."""
    eval_config = {**trackeval.Evaluator.get_default_eval_config(), "USE_PARALLEL": False, "PRINT_RESULTS": False,
                   "PRINT_CONFIG": False, "TIME_PROGRESS": False, "PLOT_CURVES": False}
    dataset_config = {**trackeval.datasets.MotChallenge2DBox.get_default_dataset_config(),
                      "GT_FOLDER": str(DATA_ROOT / "train"), "TRACKERS_FOLDER": str(OUT / "trackers"),
                      "TRACKERS_TO_EVAL": tracker_names, "OUTPUT_FOLDER": str(OUT / "eval"), "BENCHMARK": "MOT17",
                      "SPLIT_TO_EVAL": "train", "SKIP_SPLIT_FOL": True, "DO_PREPROC": True,
                      "SEQ_INFO": {f"{v}-FRCNN": None for v in video_names}, "PRINT_CONFIG": False}
    metrics = [m({"THRESHOLD": 0.5, "PRINT_CONFIG": False}) for m in
               [trackeval.metrics.HOTA, trackeval.metrics.CLEAR, trackeval.metrics.Identity]]
    results, _ = trackeval.Evaluator(eval_config).evaluate([trackeval.datasets.MotChallenge2DBox(dataset_config)], metrics)
    rows = []
    for name in tracker_names:
        for v in video_names:
            r = results["MotChallenge2DBox"][name][f"{v}-FRCNN"]["pedestrian"]
            rows.append({"tracker": name, "video": v, "HOTA": 100 * r["HOTA"]["HOTA"].mean(),
                         "MOTA": 100 * r["CLEAR"]["MOTA"], "IDF1": 100 * r["Identity"]["IDF1"],
                         "ID_switches": int(r["CLEAR"]["IDSW"]), "false_positives": int(r["CLEAR"]["CLR_FP"]),
                         "false_negatives": int(r["CLEAR"]["CLR_FN"])})
    return pd.DataFrame(rows)

## M.5 Checks before training
Each line must be True; the notebook stops otherwise. In the overfitting test the box loss can't reach 0 (binary
cross-entropy with a target like 0.74 is at least that target's entropy), so the total can only fall by about 91%.

In [ ]:
EXPECTED = [((16, 1088, 1920), 448), ((16, 1088, 1920), 2320), ((16, 1088, 1920), 2320), ((16, 272, 480), 0),
            ((32, 272, 480), 4640), ((32, 272, 480), 9248), ((32, 272, 480), 9248), ((32, 68, 120), 0),
            ((64, 68, 120), 18496), ((64, 68, 120), 36928), ((64, 68, 120), 36928), ((64, 17, 30), 0),
            ((32640,), 0), ((1024,), 33424384), ((1024,), 1049600), ((2550,), 2613750), ((64,), 384),
            ((64,), 33472), ((64,), 33472), ((64,), 33472), ((64,), 64)]
checks = {
    "GPU is the RTX 5070": "RTX 5070" in torch.cuda.get_device_name(0),
    "every parameter is on cuda": all(p.is_cuda for p in model.parameters()),
    "architecture matches the plan, 37,309,174 weights": [(r[1], r[2]) for r in rows] == EXPECTED
                                                         and weights(model) == 37_309_174,
    "every box is assigned, lost to a collision or centred outside": all(
        c["assigned"] + c["lost to collision"] + c["centre outside"] == c["boxes"] for _, c in target_counts.iterrows()),
    f"targets equal the boxes.csv boxes (largest difference {largest_difference})": largest_difference <= 1e-6,
}

# Gradient accumulation: 1 batch of 8 pairs vs 2 micro-batches of 4 (16 frames at once is more than 12 GB:
# Windows borrows shared system memory, which is slow but fine for one check).
pairs8 = [train_pairs[i] for i in range(0, 800, 100)]
gradients = lambda m: torch.cat([p.grad.flatten() for p in m.parameters() if p.grad is not None])
seed_everything()
model = TrackerNet().to(DEVICE)
(sum(pair_losses(model, *mot17_data.collate(pairs8))[:3]).sum() / 8).backward()
one_batch = gradients(model).clone()
model.zero_grad()
for half in [pairs8[:4], pairs8[4:]]:
    (sum(pair_losses(model, *mot17_data.collate(half))[:3]).sum() / 8).backward()
difference = ((one_batch - gradients(model)).norm() / one_batch.norm()).item()
checks[f"gradient accumulation: 1 x 8 = 2 x 4 pairs (relative difference {difference:.1e})"] = difference < 1e-5

# Overfitting test: 4 pairs, 200 steps, learning rate 0.001.
seed_everything()
model = TrackerNet().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
batch4 = mot17_data.collate([train_pairs[i] for i in range(4)])
losses = []
for step in range(200):
    optimizer.zero_grad()
    person, box, ident, _, _ = pair_losses(model, *batch4)
    loss = (person + box + ident).mean()
    loss.backward()
    optimizer.step()
    losses.append([loss.item(), person.mean().item(), box.mean().item(), ident.mean().item()])
model.eval()
with torch.no_grad():
    *_, correct, total = pair_losses(model, *batch4)
fall = 1 - losses[-1][0] / losses[0][0]
print("loss, person, box, ID at steps 1 and 200:", np.round(losses[0], 3), np.round(losses[-1], 3))
checks[f"overfitting: loss falls by more than 90% ({100 * fall:.1f}%)"] = fall > 0.9
checks[f"overfitting: ID accuracy 100% ({correct} of {total})"] = correct == total

for name, passed in checks.items():
    print(passed, "-", name)
del model, optimizer
torch.cuda.empty_cache()
assert all(checks.values()), "A check failed: stop and report to Rowan"

## M.6 First iteration: batch 64, learning rate 0.001, 16 epochs
Trained to 24 epochs because it is also part of the grid; the first-iteration model is the epoch-16 checkpoint
(no setting depends on the total number of epochs).

In [ ]:
first_history = train_run(BATCH_SIZE, LEARNING_RATE)
hours = first_history["seconds"].mean() * MAX_EPOCHS / 3600
print(f"One run takes about {hours:.1f} h; the grid ({len(GRID)} runs) about {len(GRID) * hours:.1f} h")

## M.7 Curves
Training, validation and testing. The test curve is only shown; it's never used to choose anything.

In [ ]:
SPLITS = {"train": ("training", "#2a78d6"), "val": ("validation", "#eb6834"), "test": ("testing", "#1baf7a")}


def plot_curves(history, epochs, name):
    h = history[history["epoch"] <= epochs]
    for column, label in [("loss", "total loss"), ("id_acc", "ID accuracy")]:
        fig, ax = plt.subplots(figsize=(12, 4.5))
        for split, (split_name, colour) in SPLITS.items():
            ax.plot(h["epoch"], h[f"{split}_{column}"], color=colour, marker="o", label=split_name)
        ax.set(xlabel="epoch", ylabel=label, title=f"{name}: {label}")
        ax.legend()
        ax.grid(alpha=0.3)
        fig.savefig(OUT / "figures" / f"{name}_{column}.png", dpi=100, bbox_inches="tight")
        plt.show()
    fig, ax = plt.subplots(figsize=(12, 4.5))
    for part, (_, colour) in zip(["person", "box", "id"], SPLITS.values()):
        ax.plot(h["epoch"], h[f"train_{part}"], color=colour, label=f"{part} loss, training")
        ax.plot(h["epoch"], h[f"val_{part}"], color=colour, linestyle="--", label=f"{part} loss, validation")
    ax.set(xlabel="epoch", ylabel="loss", title=f"{name}: loss parts")
    ax.legend(ncol=3)
    ax.grid(alpha=0.3)
    fig.savefig(OUT / "figures" / f"{name}_loss_parts.png", dpi=100, bbox_inches="tight")
    plt.show()


plot_curves(first_history, EPOCHS, "first_iteration")

## M.8 Sample image: MOT17-13, frame 300
All on the 1920 × 1088 model input: (1) the input with the cleaned ground-truth boxes (the targets; the model gets
only the image), (2) the boxes of cells with *person here?* > 0.5 and their probability, (3) the same boxes with
their IDs from tracking MOT17-13 from frame 1. Each panel is shown large; the three side by side are saved.

In [ ]:
def sample_figure(found, name, frame=300):
    data = MOT17Frames("test")                          # the test split is MOT17-13
    item = data[int(data.frames.index[data.frames["frame"] == frame][0])]
    image = (item["image"] * torch.tensor(data.std)[:, None, None] + torch.tensor(data.mean)[:, None, None])
    image = image.clamp(0, 1).permute(1, 2, 0)          # normalization undone for display
    gt = pd.DataFrame(item["boxes"].numpy(), columns=["cx", "cy", "w", "h"])
    here = found[(found["video"] == "MOT17-13") & (found["frame"] == frame)]
    stages = [(gt, "#2a78d6", None, f"1. Model input + {len(gt)} cleaned ground-truth boxes (targets, not given to the model)"),
              (here, "#eb6834", [f"{c:.2f}" for c in here["conf"]], f"2. After the fully connected layers: {len(here)} cells with person here? > 0.5"),
              (here, None, [str(i) for i in here["id"]], "3. Final output: the same boxes with their IDs")]

    def draw(ax, k):
        b, colour, labels, title = stages[k]
        ax.imshow(image)
        for j, r in enumerate(b.itertuples()):
            c = colour or plt.cm.tab20(r.id % 20)
            left, top = (r.cx - r.w / 2) * 1920, (r.cy - r.h / 2) * 1088
            ax.add_patch(Rectangle((left, top), r.w * 1920, r.h * 1088, fill=False, edgecolor=c, linewidth=2))
            if labels:
                ax.text(left, top - 4, labels[j], color=c, fontsize=9, fontweight="bold")
        ax.set_title(title)
        ax.axis("off")

    for k in range(3):
        fig, ax = plt.subplots(figsize=(16, 9.4))
        draw(ax, k)
        plt.show()
    fig, axes = plt.subplots(1, 3, figsize=(30, 6.4))
    for k in range(3):
        draw(axes[k], k)
    fig.savefig(OUT / "figures" / f"{name}_sample.png", dpi=100, bbox_inches="tight")
    plt.close(fig)


first_model = load_model(BATCH_SIZE, LEARNING_RATE, EPOCHS)
first_test = track(first_model, "test", "first_iteration")
sample_figure(first_test, "first_iteration")

## M.9 Grid search (validation only)
Every (batch size, learning rate) run trains 24 epochs; epochs 8, 16 and 24 are compared. The best setting has the
lowest **validation** total loss. Only the first-iteration and best checkpoints are kept.

In [ ]:
grid_hours = len(GRID) * hours
assert grid_hours <= 10 or ALLOW_LONG_GRID, (f"The grid would take about {grid_hours:.1f} h: ask Rowan, then use the "
                                             "4-run OPTION in M.0 or set ALLOW_LONG_GRID = True")
grid = []
for batch_size, lr in GRID:
    history = train_run(batch_size, lr).set_index("epoch")
    for epochs in (8, 16, 24):
        grid.append({"batch_size": batch_size, "lr": lr, "epochs": epochs, "val_loss": history.loc[epochs, "val_loss"],
                     "val_id_acc": history.loc[epochs, "val_id_acc"], "train_loss": history.loc[epochs, "train_loss"]})
    best = pd.DataFrame(grid).sort_values("val_loss").iloc[0]                  # validation numbers only
    BEST = (int(best["batch_size"]), float(best["lr"]), int(best["epochs"]))   # (batch size, learning rate, epochs)
    keep = {f"bs{BATCH_SIZE}_lr{LEARNING_RATE:g}_ep{EPOCHS}.pt", f"bs{BEST[0]}_lr{BEST[1]:g}_ep{BEST[2]}.pt"}
    for file in (OUT / "checkpoints").glob("*.pt"):
        if file.name not in keep:
            file.unlink()
grid = pd.DataFrame(grid)
display(grid.sort_values("val_loss"))
print(f"Best: batch {BEST[0]}, learning rate {BEST[1]:g}, {BEST[2]} epochs")

## M.10 Final results

In [ ]:
best_history = pd.read_csv(OUT / "history" / f"bs{BEST[0]}_lr{BEST[1]:g}.csv")
plot_curves(best_history, BEST[2], "best")
best_model = load_model(*BEST)
best_test = track(best_model, "test", "best")
sample_figure(best_test, "best")
track(first_model, "val", "first_iteration")
track(best_model, "val", "best")

scores = score(["first_iteration", "best"])
display(scores.round(1))
deepsort = pd.read_csv(mot17_data.PROJECT_ROOT / "outputs" / "deepsort" / "results.csv").assign(tracker="DeepSORT baseline")
COLUMNS = ["tracker", "HOTA", "MOTA", "IDF1", "ID_switches", "false_positives", "false_negatives"]
comparison = pd.concat([scores, deepsort])
comparison = comparison[comparison["video"] == "MOT17-13"][COLUMNS]
display(comparison.round(1))

In [ ]:
ours, base = comparison.set_index("tracker").loc["best"], comparison.set_index("tracker").loc["DeepSORT baseline"]
meaning = (f"On the test video MOT17-13 the best model (batch {BEST[0]}, learning rate {BEST[1]:g}, {BEST[2]} "
           f"epochs) scores HOTA {ours.HOTA:.1f}, MOTA {ours.MOTA:.1f} and IDF1 {ours.IDF1:.1f}; the DeepSORT baseline scores "
           f"HOTA {base.HOTA:.1f}, MOTA {base.MOTA:.1f} and IDF1 {base.IDF1:.1f}. By design a cell holds at most one person: "
           f"{target_counts.loc['test', 'lost to collision']:,} of the test video's {target_counts.loc['test', 'boxes']:,} boxes "
           "share a cell with a more visible person, so even a perfect model of this design can't find them.")
report = f"""# Model report

## Architecture
```
{architecture.to_string(index=False)}
```
Total weights: {architecture['weights'].sum():,}

## Settings
Batch size {BATCH_SIZE}, learning rate {LEARNING_RATE}, {EPOCHS} epochs (first iteration); Adam; micro-batch {MICRO_BATCH};
grid {ROWS} x {COLS}; person threshold {PERSON_THRESHOLD}; pos_weight {POS_WEIGHT.item():.2f}; seed 0; {torch.cuda.get_device_name(0)}.

## Training targets
```
{target_counts.to_string()}
```

## Grid search (validation total loss, lowest first)
```
{grid.sort_values("val_loss").round(4).to_string(index=False)}
```

## Comparison on MOT17-13 (test)
```
{comparison.round(1).to_string(index=False)}
```

Curves and sample images: `figures/`.

## What it means
{meaning}
"""
(OUT / "model_report.md").write_text(report, encoding="utf-8")
print(meaning)

## M.11 Checks after training

In [ ]:
for name in ["first_iteration", "best"]:
    for video in ["MOT17-10", "MOT17-13"]:
        t = pd.read_csv(OUT / "trackers" / name / "data" / f"{video}-FRCNN.txt", header=None,
                        names=["frame", "id", "left", "top", "width", "height", "conf", "x", "y", "z"])
        valid = (t["frame"].between(1, videos.loc[video, "length"]).all() and (t["id"] > 0).all()
                 and not t.duplicated(["frame", "id"]).any()                       # no ID twice in one frame
                 and (t["left"] + t["width"] / 2).between(0, videos.loc[video, "width"]).all()
                 and t["width"].max() > 1)                                         # pixels, not fractions
        print(f"{name} {video} tracker file valid: {valid} ({len(t):,} boxes)")
print("gt.txt unchanged:", all((DATA_ROOT / videos.loc[v, "folder"] / "gt" / "gt.txt").read_bytes() == gt_before[v]
                               for v in videos.index))
print("every run has training, validation and test curves:",
      all(pd.read_csv(OUT / "history" / f"bs{b}_lr{lr:g}.csv")[["train_loss", "val_loss", "test_loss"]].notna().all().all()
          for b, lr in GRID))
lowest = grid.loc[grid["val_loss"].idxmin()]
print("best setting chosen on validation loss only:",
      (int(lowest["batch_size"]), float(lowest["lr"]), int(lowest["epochs"])) == BEST)